# 03 · Panel de visualizaciones publicables

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from simel_client import SIMELClient

plt.rcParams.update({'font.family':'sans-serif','axes.titlesize':12})
client = SIMELClient()
def cargar(ds):
    p = f'data/{ds}.csv'
    return pd.read_csv(p) if os.path.exists(p) else client.get(ds)

tnr_sexo = cargar('DF_HRS_TNR_SEXO')
tnr_edad = cargar('DF_HRS_TNR_SEXO_EDAD')
fdt      = cargar('DF_FDT_SEXO')
ROJO, AZUL = '#c0392b', '#2e86c1'
os.makedirs('outputs/figures', exist_ok=True)
print('Listo.')

## Panel resumen — 3 figuras clave

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# A: barras 2015 vs 2023
ax = axes[0]
hm = tnr_sexo[tnr_sexo['SEXO'].isin(['M','F'])]
anios = sorted(hm['AÑO'].unique()); x=np.arange(len(anios)); w=0.35
for i,(sexo,color,label) in enumerate([('F',ROJO,'Mujeres'),('M',AZUL,'Hombres')]):
    vals = hm[hm['SEXO']==sexo].set_index('AÑO')['OBS_VALUE']
    b = ax.bar(x+i*w,[vals.get(a,np.nan) for a in anios],w,color=color,label=label,edgecolor='white',alpha=0.9)
    ax.bar_label(b,fmt='%.1fh',padding=2,fontsize=8)
ax.set_xticks(x+w/2); ax.set_xticklabels(anios); ax.set_ylabel('Horas/día')
ax.set_title('A · TNR por sexo (2015 vs 2023)', fontweight='bold'); ax.legend(fontsize=8)
sns.despine(ax=ax)

# B: ciclo de vida
ax2 = axes[1]
edad_col = next((c for c in tnr_edad.columns if 'EDAD' in c.upper()), tnr_edad.columns[4])
e_max = tnr_edad['AÑO'].max()
for sexo,color,label in [('F',ROJO,'Mujeres'),('M',AZUL,'Hombres')]:
    sub = tnr_edad[(tnr_edad['SEXO']==sexo)&(tnr_edad['AÑO']==e_max)&(tnr_edad[edad_col]!='_T')].sort_values(edad_col)
    if len(sub):
        ax2.plot(range(len(sub)),sub['OBS_VALUE'],'o-',color=color,label=label,linewidth=2,markersize=6)
        ax2.set_xticks(range(len(sub))); ax2.set_xticklabels(sub[edad_col].values,rotation=30,ha='right',fontsize=7)
ax2.set_ylabel('Horas/día'); ax2.legend(fontsize=8)
ax2.set_title(f'B · Ciclo de vida del cuidado — {e_max}', fontweight='bold')
sns.despine(ax=ax2)

# C: participación laboral regional
ax3 = axes[2]
fdt_max = fdt['AÑO'].max()
fdt_r = fdt[(fdt['AÑO']==fdt_max)&(fdt['AREA_REF']!='_T')&(fdt['SEXO'].isin(['M','F']))]
if len(fdt_r):
    piv = fdt_r.pivot_table(index='REGION',columns='SEXO',values='OBS_VALUE')
    if 'F' in piv and 'M' in piv:
        piv['ratio'] = piv['F']/piv['M']
        piv = piv.sort_values('ratio')
        cols_c = [ROJO if r<0.8 else '#e67e22' if r<0.9 else '#27ae60' for r in piv['ratio']]
        ax3.barh(piv.index, piv['ratio'], color=cols_c, edgecolor='white')
        ax3.axvline(1,color='gray',linestyle='--',linewidth=1)
        ax3.set_xlabel('Ratio FDT M/H')
ax3.set_title(f'C · Participación laboral — {fdt_max}', fontweight='bold')
sns.despine(ax=ax3)

fig.suptitle('Economía del Cuidado en Chile — Resumen visual', fontsize=14, fontweight='bold', y=1.01)
fig.text(0.5,-0.01,'Fuente: SIMEL-INE · ENUT 2015/2023 · ENE · Elaboración propia',ha='center',fontsize=8,color='gray')
plt.tight_layout()
plt.savefig('outputs/figures/panel_resumen.png', dpi=150, bbox_inches='tight')
plt.show()
print('Panel guardado.')

## Conclusiones

1. **La brecha se redujo pero no redistribuyó**: bajaron las horas de las mujeres (más trabajo remunerado), no subieron las de los hombres.
2. **La carga es más pesada en los años reproductivos** (30–44), afectando la acumulación de experiencia y pensiones.
3. **La participación laboral femenina es menor en todas las regiones** — el trabajo de cuidados actúa como barrera estructural.

> El trabajo no remunerado equivale a varios meses laborales por año — invisible en el PIB, pero central para sostener la economía.